# Topic 5 — BigQuery: Ask Hacker News (Mini Agent)

A public dataset (real scale, free to query up to 1 TiB/month), the `LIMIT`-doesn't-reduce-cost lesson, and a small function-calling agent — same plain SDK pattern as Module 3, no framework.

In [ ]:
from google.cloud import bigquery
from setup import PROJECT_ID, genai_client, MODEL_FLASH
from google.genai import types

bq_client = bigquery.Client(project=PROJECT_ID)

### The pricing lesson first — always check bytes scanned BEFORE running

In [ ]:
sql = """
SELECT title, score
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
ORDER BY score DESC
LIMIT 10
"""

job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run_job = bq_client.query(sql, job_config=job_config)
print(f"This query will scan {dry_run_job.total_bytes_processed / 1e9:.2f} GB")
print("Notice: LIMIT 10 does NOT change this number - it only limits what's displayed.")

### Now actually run it

In [ ]:
results = bq_client.query(sql).result()
for row in results:
    print(row.score, "-", row.title)

### The mini agent — natural language question to SQL to answer

Gemini is given the table schema and asked to write the SQL itself.

In [ ]:
HN_SCHEMA_PROMPT = """
You have access to a BigQuery table: `bigquery-public-data.hacker_news.full`
Key columns: title (STRING), url (STRING), text (STRING), by (STRING, the author),
score (INTEGER), time_ts (TIMESTAMP), type (STRING: 'story', 'comment', etc.), id (INTEGER).

When asked a question, call run_bigquery_sql with a single valid BigQuery Standard SQL
query against this table. Keep results small (use LIMIT or aggregate with COUNT/AVG).
"""

def run_bigquery_sql(sql: str) -> str:
    """Execute a BigQuery Standard SQL query against bigquery-public-data.hacker_news.full and return the results."""
    print(f"\n[System] Running SQL:\n{sql}\n")
    rows = list(bq_client.query(sql).result())
    return str(rows[:20])  # cap what comes back to the model

In [ ]:
run_sql_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="run_bigquery_sql",
        description="Execute a BigQuery Standard SQL query against the Hacker News public dataset",
        parameters={
            "type": "object",
            "properties": {"sql": {"type": "string", "description": "A valid BigQuery Standard SQL query"}},
            "required": ["sql"],
        },
    )
])

question = "How many Hacker News stories in 2023 had 'GPT' in the title?"

response = genai_client.models.generate_content(
    model=MODEL_FLASH,
    contents=HN_SCHEMA_PROMPT + "\n\nQuestion: " + question,
    config=types.GenerateContentConfig(tools=[run_sql_tool]),
)

call = response.candidates[0].content.parts[0].function_call
sql_result = run_bigquery_sql(**call.args)

final = genai_client.models.generate_content(
    model=MODEL_FLASH,
    contents=[
        HN_SCHEMA_PROMPT + "\n\nQuestion: " + question,
        response.candidates[0].content,
        types.Content(parts=[types.Part.from_function_response(
            name="run_bigquery_sql", response={"result": sql_result},
        )]),
    ],
    config=types.GenerateContentConfig(tools=[run_sql_tool]),
)

print(final.text)